# Feature Engineering Notebook
### Sales Forecasting & Gen AI Project

## 1. Install & Import Libraries

In [1]:
# !pip install pandas numpy sqlalchemy pymysql


In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

C:\Users\shahi\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\shahi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

## 2. MySQL Connection

In [4]:
from urllib.parse import quote_plus

In [5]:
DB_USER = "root"
DB_PASSWORD =quote_plus("@ry@nSh1")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "ai_sales_analysis"


In [6]:
connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Connection test
with engine.connect() as conn:
    result = conn.execute(text("SELECT VERSION();"))
    print("Connected! MySQL version:", result.fetchone()[0])

Connected! MySQL version: 8.0.32


## 3. Load Raw Tables

In [7]:
sales = pd.read_sql("SELECT * FROM sales_transactions", engine)
customers = pd.read_sql("SELECT * FROM customers", engine)
products = pd.read_sql("SELECT * FROM products", engine)
targets = pd.read_sql("SELECT * FROM targets", engine)

In [8]:
import pandas, sqlalchemy
print(pandas.__version__, sqlalchemy.__version__)

3.0.5 2.0.52


In [9]:
# pip install --upgrade sqlalchemy

In [10]:
sales['txn_date'] = pd.to_datetime(sales['txn_date'])
sales['year'] = sales['txn_date'].dt.year
sales['month'] = sales['txn_date'].dt.month

print("sales:", sales.shape, "| customers:", customers.shape,
      "| products:", products.shape, "| targets:", targets.shape)
sales.head()

sales: (8994, 11) | customers: (251, 6) | products: (39, 5) | targets: (36, 3)


,transaction_id,txn_date,product_id,customer_id,region,quantity,unit_price,discount,revenue,year,month
0,T100000,2023-10-13,P0029,C00010,south,1,12773.07,0.0,12773.07,2023,10
1,T100001,2023-05-29,P0011,C00238,west,6,8863.25,0.0,53179.50,2023,5
2,T100002,2024-10-02,P0010,C00051,east,3,8164.43,0.0,24493.29,2024,10
3,T100003,2025-09-04,P0030,C00039,north,3,6824.34,0.0,20473.02,2025,9
4,T100004,2024-03-31,P0009,C00126,west,6,11247.70,0.0,67486.20,2024,3


## 4. Monthly Base Table

In [11]:
monthly = (
    sales
    .groupby(['year', 'month'], as_index=False)
    .agg(total_revenue=('revenue', 'sum'),
         order_count=('transaction_id', 'count'),
         active_customers=('customer_id', 'nunique'))
    .sort_values(['year', 'month'])
    .reset_index(drop=True)
)

monthly['quarter'] = ((monthly['month'] - 1) // 3) + 1
monthly.head(12)


,year,month,total_revenue,order_count,active_customers,quarter
0,2023,1,6282761.26,243,153,1
1,2023,2,5584972.24,198,124,1
2,2023,3,7225276.49,244,150,1
3,2023,4,10616603.50,230,147,2
4,2023,5,12021876.44,262,166,2
5,2023,6,7846609.33,236,155,2
6,2023,7,9589366.48,249,160,3
7,2023,8,11182936.14,265,164,3
8,2023,9,7807263.08,258,154,3
9,2023,10,9224626.01,242,151,4


## 5. Lag & Rolling Features

- `revenue_lag_1`, `revenue_lag_3`, `revenue_lag_12`
- `rolling_mean_3`, `rolling_mean_6`, `rolling_std_3`
- `mom_growth_pct`, `yoy_growth_pct`


In [12]:
monthly['revenue_lag_1']  = monthly['total_revenue'].shift(1)
monthly['revenue_lag_3']  = monthly['total_revenue'].shift(3)
monthly['revenue_lag_12'] = monthly['total_revenue'].shift(12)

monthly['rolling_mean_3'] = monthly['total_revenue'].rolling(window=3).mean()
monthly['rolling_mean_6'] = monthly['total_revenue'].rolling(window=6).mean()
monthly['rolling_std_3']  = monthly['total_revenue'].rolling(window=3).std()

monthly['mom_growth_pct'] = (
    (monthly['total_revenue'] - monthly['revenue_lag_1']) / monthly['revenue_lag_1'] * 100
).round(2)

monthly['yoy_growth_pct'] = (
    (monthly['total_revenue'] - monthly['revenue_lag_12']) / monthly['revenue_lag_12'] * 100
).round(2)

# Seasonal flag — Oct/Nov/Dec as festive months (adjust as per your actual seasonality)
monthly['is_festive_month'] = monthly['month'].isin([10, 11, 12]).astype(int)

monthly.head(15)


,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month
0,2023,1,6282761.26,243,153,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2023,2,5584972.24,198,124,1,6282761.26,NaN,NaN,NaN,NaN,NaN,-11.11,NaN,0
2,2023,3,7225276.49,244,150,1,5584972.24,NaN,NaN,6.364337e+06,NaN,8.231892e+05,29.37,NaN,0
3,2023,4,10616603.50,230,147,2,7225276.49,6282761.26,NaN,7.808951e+06,NaN,2.566093e+06,46.94,NaN,0
4,2023,5,12021876.44,262,166,2,10616603.50,5584972.24,NaN,9.954585e+06,NaN,2.465876e+06,13.24,NaN,0
5,2023,6,7846609.33,236,155,2,12021876.44,7225276.49,NaN,1.016170e+07,8.263017e+06,2.124481e+06,-34.73,NaN,0
6,2023,7,9589366.48,249,160,3,7846609.33,10616603.50,NaN,9.819284e+06,8.814117e+06,2.097108e+06,22.21,NaN,0
7,2023,8,11182936.14,265,164,3,9589366.48,12021876.44,NaN,9.539637e+06,9.747111e+06,1.668719e+06,16.62,NaN,0
8,2023,9,7807263.08,258,154,3,11182936.14,7846609.33,NaN,9.526522e+06,9.844109e+06,1.688714e+06,-30.19,NaN,0
9,2023,10,9224626.01,242,151,4,7807263.08,9589366.48,NaN,9.404942e+06,9.612113e+06,1.695045e+06,18.15,NaN,1


## 6. Discount & Order-Value Features


In [13]:
discount_features = (
    sales
    .groupby(['year', 'month'], as_index=False)
    .agg(avg_discount=('discount', 'mean'))
)
discount_features['avg_discount'] = discount_features['avg_discount'].round(3)

monthly = monthly.merge(discount_features, on=['year', 'month'], how='left')
monthly['avg_order_value'] = (monthly['total_revenue'] / monthly['order_count']).round(2)
monthly.head()


,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month,avg_discount,avg_order_value
0,2023,1,6282761.26,243,153,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.029,25854.98
1,2023,2,5584972.24,198,124,1,6282761.26,NaN,NaN,NaN,NaN,NaN,-11.11,NaN,0,0.040,28206.93
2,2023,3,7225276.49,244,150,1,5584972.24,NaN,NaN,6.364337e+06,NaN,8.231892e+05,29.37,NaN,0,0.040,29611.79
3,2023,4,10616603.50,230,147,2,7225276.49,6282761.26,NaN,7.808951e+06,NaN,2.566093e+06,46.94,NaN,0,0.033,46159.15
4,2023,5,12021876.44,262,166,2,10616603.50,5584972.24,NaN,9.954585e+06,NaN,2.465876e+06,13.24,NaN,0,0.032,45885.02


## 7. Repeat-Customer Rate


In [14]:
sales_sorted = sales.sort_values('txn_date')
first_purchase = sales_sorted.groupby('customer_id')['txn_date'].min().rename('first_purchase_date')
sales_with_first = sales.merge(first_purchase, on='customer_id', how='left')

sales_with_first['is_repeat'] = (
    sales_with_first['txn_date'] > sales_with_first['first_purchase_date']
).astype(int)

repeat_features = (
    sales_with_first
    .groupby(['year', 'month'], as_index=False)
    .agg(repeat_customer_rate=('is_repeat', 'mean'))
)
repeat_features['repeat_customer_rate'] = (repeat_features['repeat_customer_rate'] * 100).round(2)

monthly = monthly.merge(repeat_features, on=['year', 'month'], how='left')
monthly.head()


,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month,avg_discount,avg_order_value,repeat_customer_rate
0,2023,1,6282761.26,243,153,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.029,25854.98,35.39
1,2023,2,5584972.24,198,124,1,6282761.26,NaN,NaN,NaN,NaN,NaN,-11.11,NaN,0,0.040,28206.93,76.77
2,2023,3,7225276.49,244,150,1,5584972.24,NaN,NaN,6.364337e+06,NaN,8.231892e+05,29.37,NaN,0,0.040,29611.79,86.89
3,2023,4,10616603.50,230,147,2,7225276.49,6282761.26,NaN,7.808951e+06,NaN,2.566093e+06,46.94,NaN,0,0.033,46159.15,94.35
4,2023,5,12021876.44,262,166,2,10616603.50,5584972.24,NaN,9.954585e+06,NaN,2.465876e+06,13.24,NaN,0,0.032,45885.02,98.09


## 8. Top-Category Revenue Share


In [15]:
sales_products = sales.merge(products[['product_id', 'category']], on='product_id', how='left')

category_monthly = (
    sales_products
    .groupby(['year', 'month', 'category'], as_index=False)['revenue']
    .sum()
)

top_category_share = (
    category_monthly
    .sort_values('revenue', ascending=False)
    .groupby(['year', 'month'], as_index=False)
    .first()
    .rename(columns={'category': 'top_category', 'revenue': 'top_category_revenue'})
)

month_totals = category_monthly.groupby(['year', 'month'], as_index=False)['revenue'].sum() \
                                 .rename(columns={'revenue': 'month_total_revenue'})

top_category_share = top_category_share.merge(month_totals, on=['year', 'month'], how='left')
top_category_share['top_category_share'] = (
    top_category_share['top_category_revenue'] / top_category_share['month_total_revenue']
).round(3)

monthly = monthly.merge(
    top_category_share[['year', 'month', 'top_category', 'top_category_share']],
    on=['year', 'month'], how='left'
)
monthly.head()


,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month,avg_discount,avg_order_value,repeat_customer_rate,top_category,top_category_share
0,2023,1,6282761.26,243,153,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.029,25854.98,35.39,home & kitchen,0.297
1,2023,2,5584972.24,198,124,1,6282761.26,NaN,NaN,NaN,NaN,NaN,-11.11,NaN,0,0.040,28206.93,76.77,home & kitchen,0.511
2,2023,3,7225276.49,244,150,1,5584972.24,NaN,NaN,6.364337e+06,NaN,8.231892e+05,29.37,NaN,0,0.040,29611.79,86.89,home & kitchen,0.423
3,2023,4,10616603.50,230,147,2,7225276.49,6282761.26,NaN,7.808951e+06,NaN,2.566093e+06,46.94,NaN,0,0.033,46159.15,94.35,sports,0.476
4,2023,5,12021876.44,262,166,2,10616603.50,5584972.24,NaN,9.954585e+06,NaN,2.465876e+06,13.24,NaN,0,0.032,45885.02,98.09,grocery,0.417


## 9. Bring in `target_revenue` (for gap analysis later — NOT a model input feature)

In [16]:
monthly = monthly.merge(targets, on=['year', 'month'], how='left')
monthly.head()


,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month,avg_discount,avg_order_value,repeat_customer_rate,top_category,top_category_share,target_revenue
0,2023,1,6282761.26,243,153,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.029,25854.98,35.39,home & kitchen,0.297,NaN
1,2023,2,5584972.24,198,124,1,6282761.26,NaN,NaN,NaN,NaN,NaN,-11.11,NaN,0,0.040,28206.93,76.77,home & kitchen,0.511,NaN
2,2023,3,7225276.49,244,150,1,5584972.24,NaN,NaN,6.364337e+06,NaN,8.231892e+05,29.37,NaN,0,0.040,29611.79,86.89,home & kitchen,0.423,NaN
3,2023,4,10616603.50,230,147,2,7225276.49,6282761.26,NaN,7.808951e+06,NaN,2.566093e+06,46.94,NaN,0,0.033,46159.15,94.35,sports,0.476,NaN
4,2023,5,12021876.44,262,166,2,10616603.50,5584972.24,NaN,9.954585e+06,NaN,2.465876e+06,13.24,NaN,0,0.032,45885.02,98.09,grocery,0.417,NaN


## 10. Final Feature Table — Preview & Null Check

In [17]:
print("Shape:", monthly.shape)
print("\nNull counts:\n", monthly.isna().sum())
monthly.tail(10)


Shape: (36, 21)

Null counts:
 year                     0
month                    0
total_revenue            0
order_count              0
active_customers         0
quarter                  0
revenue_lag_1            1
revenue_lag_3            3
revenue_lag_12          12
rolling_mean_3           2
rolling_mean_6           5
rolling_std_3            2
mom_growth_pct           1
yoy_growth_pct          12
is_festive_month         0
avg_discount             0
avg_order_value          0
repeat_customer_rate     0
top_category             0
top_category_share       0
target_revenue          12
dtype: int64


,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month,avg_discount,avg_order_value,repeat_customer_rate,top_category,top_category_share,target_revenue
26,2025,3,8361547.00,244,152,1,6571783.42,16146316.86,8084247.51,7.107128e+06,1.014212e+07,1.090236e+06,27.23,3.43,0,0.096,34268.64,100.0,home & kitchen,0.428,2875000.0
27,2025,4,15696457.88,276,170,2,8361547.00,6388053.41,8383036.20,1.020993e+07,1.083634e+07,4.835009e+06,87.72,87.24,0,0.036,56871.22,100.0,home & kitchen,0.626,2875000.0
28,2025,5,10118594.94,276,165,2,15696457.88,6571783.42,8797381.85,1.139220e+07,1.054713e+07,3.829723e+06,-35.54,15.02,0,0.031,36661.58,100.0,home & kitchen,0.384,2875000.0
29,2025,6,8326845.06,256,157,2,10118594.94,8361547.00,8429387.84,1.138063e+07,9.243880e+06,3.843482e+06,-17.71,-1.22,0,0.075,32526.74,100.0,home & kitchen,0.405,2875000.0
30,2025,7,12490297.99,276,165,3,8326845.06,15696457.88,7130056.75,1.031191e+07,1.026092e+07,2.088448e+06,50.00,75.18,0,0.122,45254.70,100.0,electronics,0.334,2875000.0
31,2025,8,12102756.18,237,154,3,12490297.99,10118594.94,9381893.40,1.097330e+07,1.118275e+07,2.300074e+06,-3.10,29.00,0,0.031,51066.48,100.0,sports,0.546,2875000.0
32,2025,9,10132338.42,270,158,3,12102756.18,8326845.06,7895263.74,1.157513e+07,1.147788e+07,1.264431e+06,-16.28,28.33,0,0.034,37527.18,100.0,home & kitchen,0.334,2875000.0
33,2025,10,11242309.04,238,156,4,10132338.42,12490297.99,11531126.90,1.115913e+07,1.073552e+07,9.878386e+05,10.95,-2.50,1,0.036,47236.59,100.0,home & kitchen,0.411,4025000.0
34,2025,11,12207746.02,224,157,4,11242309.04,12102756.18,11853897.33,1.119413e+07,1.108372e+07,1.038542e+06,8.59,2.99,1,0.075,54498.87,100.0,home & kitchen,0.358,4600000.0
35,2025,12,14154455.35,274,171,4,12207746.02,10132338.42,16146316.86,1.253484e+07,1.205498e+07,1.483371e+06,15.95,-12.34,1,0.039,51658.60,100.0,home & kitchen,0.301,4312000.0


## 11. Save Final Feature Table (CSV + MySQL)


In [19]:
monthly.to_csv("feature_table.csv", index=False)


monthly.to_sql("feature_table", engine, if_exists="replace", index=False)


36